# Customer Churn Prediction using Amazon SageMaker Pipelines

## End-to-End MLOps Pipeline

This project demonstrates a complete Machine Learning Operations (MLOps) workflow using Amazon SageMaker Pipelines for predicting customer churn on the IBM Telco Customer Churn dataset.

The pipeline automates the complete machine learning lifecycle:

- Data preprocessing
- Model training
- Model evaluation
- Accuracy validation
- Conditional model registration
- Deployment-ready model packaging

The project uses:

- Amazon SageMaker Pipelines
- Scikit-learn Pipeline
- ColumnTransformer
- OneHotEncoder
- XGBoost Classifier
- Model Registry
- Conditional Workflow Execution

---

## Project Structure



'''

## Pipeline Architecture

Raw Dataset
↓
ProcessingStep
↓
TrainingStep
↓
EvaluationStep
↓
ConditionStep
↓
RegisterModel

'''

## Expected Pipeline Flow

1. Read dataset from Amazon S3
2. Clean and split dataset
3. Train XGBoost model using Scikit-learn Pipeline
4. Evaluate on test dataset
5. Generate evaluation report
6. Compare accuracy against threshold
7. Register model if accuracy ≥ threshold
8. Model becomes deployment-ready

---

In [ ]:
# ==========================================================
# Install Required Packages
# ==========================================================

# Uncomment if running on a fresh notebook

# !pip install "sagemaker==2.250"
# !pip install \
# boto3 \
# pandas==2.2.2 \
# numpy==1.26.4 \
# scikit-learn==1.5.1 \
# xgboost==2.1.1 \
# joblib==1.4.2

In [ ]:
# ==========================================================
# Verify Environment
# ==========================================================

import sys
import boto3
import pandas as pd
import numpy as np
import sklearn
import xgboost
import joblib
import sagemaker

print("=" * 60)
print("Environment Information")
print("=" * 60)

print(f"Python      : {sys.version.split()[0]}")
print(f"Boto3       : {boto3.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"Numpy       : {np.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"XGBoost     : {xgboost.__version__}")
print(f"Joblib      : {joblib.__version__}")
print(f"SageMaker   : {sagemaker.__version__}")

print("=" * 60)

session = boto3.Session()

print(f"AWS Region  : {session.region_name}")

print("=" * 60)
print("Environment Ready")
print("=" * 60)

# Section 2 — Imports & Configuration

This section imports all required SageMaker SDK modules, loads the project configuration, and initializes the AWS and SageMaker sessions used throughout the pipeline.

The configuration values are maintained in `config.py`, allowing the notebook to remain clean and making environment-specific changes easy.

In [ ]:
# ==========================================================
# Standard Library Imports
# ==========================================================

import boto3

# ==========================================================
# SageMaker Sessions
# ==========================================================

from sagemaker.session import Session
from sagemaker.workflow.pipeline_context import PipelineSession

# ==========================================================
# Processing & Training
# ==========================================================

from sagemaker.processing import (
    ProcessingInput,
    ProcessingOutput,
)

from sagemaker.inputs import TrainingInput

from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn

# ==========================================================
# Pipeline Parameters
# ==========================================================

from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterFloat,
    ParameterString,
)

# ==========================================================
# Pipeline Steps
# ==========================================================

from sagemaker.workflow.steps import (
    ProcessingStep,
    TrainingStep,
)

from sagemaker.workflow.pipeline import Pipeline

# ==========================================================
# Model Evaluation
# ==========================================================

from sagemaker.workflow.functions import (
    JsonGet,
    Join,
)

from sagemaker.workflow.properties import PropertyFile

from sagemaker.model_metrics import (
    MetricsSource,
    ModelMetrics,
)

# ==========================================================
# Conditional Execution
# ==========================================================

from sagemaker.workflow.conditions import (
    ConditionGreaterThanOrEqualTo,
)

from sagemaker.workflow.condition_step import (
    ConditionStep,
)

# ==========================================================
# Model Registration
# ==========================================================

from sagemaker.workflow.step_collections import (
    RegisterModel,
)



In [ ]:
# ==========================================================
# Project Configuration
# ==========================================================

from pipeline.config import *

from pipeline.utils import (
    create_evaluation_report,
    get_accuracy_condition,
)

print("=" * 60)
print("Configuration Loaded")
print("=" * 60)

print(f"Pipeline Name           : {PIPELINE_NAME}")
print(f"Bucket                  : {BUCKET_NAME}")
print(f"Region                  : {AWS_REGION}")
print(f"Processing Instance     : {PROCESSING_INSTANCE_TYPE}")
print(f"Training Instance       : {TRAINING_INSTANCE_TYPE}")
print(f"Evaluation Instance     : {EVALUATION_INSTANCE_TYPE}")
print(f"Model Package Group     : {MODEL_PACKAGE_GROUP_NAME}")
print(f"Accuracy Threshold      : {ACCURACY_THRESHOLD}")

print("=" * 60)


In [ ]:
print(sagemaker.get_execution_role())


In [ ]:
# ==========================================================
# AWS Session
# ==========================================================

boto_session = boto3.Session(
    region_name=AWS_REGION
)

sagemaker_session = Session(
    boto_session=boto_session
)

pipeline_session = PipelineSession(
    boto_session=boto_session
)

role = ROLE

print("=" * 60)
print("AWS Session Initialized")
print("=" * 60)

print(f"Region            : {AWS_REGION}")
print(f"Execution Role    : {ROLE}")
print(f"S3 Bucket         : {BUCKET_NAME}")

print("=" * 60)

# Section 3 — Pipeline Parameters

Amazon SageMaker Pipelines allows many values to be parameterized rather than hardcoded.

This makes the pipeline reusable across different environments without modifying the source code.

The following parameters are exposed:

- Processing instance count
- Processing instance type
- Training instance type
- Model approval status
- Accuracy threshold

These parameters can be overridden when the pipeline is executed.


In [ ]:
# ==========================================================
# Pipeline Parameters
# ==========================================================

processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount",
    default_value=INSTANCE_COUNT,
)

processing_instance_type = ParameterString(
    name="ProcessingInstanceType",
    default_value=PROCESSING_INSTANCE_TYPE,
)

training_instance_type = ParameterString(
    name="TrainingInstanceType",
    default_value=TRAINING_INSTANCE_TYPE,
)

model_approval_status = ParameterString(
    name="ModelApprovalStatus",
    default_value="PendingManualApproval",
)

accuracy_threshold = ParameterFloat(
    name="AccuracyThreshold",
    default_value=ACCURACY_THRESHOLD,
)

print("=" * 60)
print("Pipeline Parameters Created")
print("=" * 60)

print(f"Processing Instance Count : {INSTANCE_COUNT}")
print(f"Processing Instance Type  : {PROCESSING_INSTANCE_TYPE}")
print(f"Training Instance Type    : {TRAINING_INSTANCE_TYPE}")
print(f"Approval Status           : PendingManualApproval")
print(f"Accuracy Threshold        : {ACCURACY_THRESHOLD}")

print("=" * 60)


# Section 4 — Processing & Training Components

The pipeline uses two SageMaker components:

### SKLearnProcessor

Used for:

- preprocessing.py
- evaluate.py

A processor launches a temporary SageMaker Processing Job.

---

### SKLearn Estimator

Used for:

- train.py

The estimator launches a SageMaker Training Job and stores the trained model in Amazon S3.

The estimator also installs all packages listed in `requirements.txt` before executing the training script.

In [ ]:
# ==========================================================
# SKLearn Processor
# ==========================================================

script_processor = SKLearnProcessor(

    framework_version=SKLEARN_FRAMEWORK_VERSION,

    role=role,

    instance_type=processing_instance_type,

    instance_count=processing_instance_count,

    sagemaker_session=pipeline_session,

)

print("=" * 60)
print("SKLearnProcessor Created")
print("=" * 60)

print(f"Framework Version : {SKLEARN_FRAMEWORK_VERSION}")
print(f"Instance Type     : {PROCESSING_INSTANCE_TYPE}")
print(f"Instance Count    : {INSTANCE_COUNT}")

print("=" * 60)

In [ ]:
# ==========================================================
# SKLearn Estimator
# ==========================================================

estimator = SKLearn(

    entry_point=TRAIN_SCRIPT,

    source_dir=SOURCE_DIR,

    role=role,

    framework_version=SKLEARN_FRAMEWORK_VERSION,

    py_version=PYTHON_VERSION,

    instance_count=INSTANCE_COUNT,

    instance_type=training_instance_type,

    output_path=MODEL_OUTPUT_URI,

    sagemaker_session=pipeline_session,

    dependencies=[
        "requirements.txt"
    ],

)

print("=" * 60)
print("SKLearn Estimator Created")
print("=" * 60)

print(f"Training Script : {TRAIN_SCRIPT}")
print(f"Source Directory: {SOURCE_DIR}")
print(f"Framework       : {SKLEARN_FRAMEWORK_VERSION}")
print(f"Python          : {PYTHON_VERSION}")
print(f"Output Path     : {MODEL_OUTPUT_URI}")

print("=" * 60)

# Section 5 — Data Preprocessing

The first pipeline step executes `preprocessing.py` using an `SKLearnProcessor`.

The preprocessing job performs the following tasks:

- Reads the raw customer churn dataset from Amazon S3
- Cleans the dataset
- Removes unnecessary columns
- Converts data types
- Handles missing values
- Removes duplicate records
- Splits the data into training and testing datasets
- Saves the processed datasets back to Amazon S3

The outputs from this step are consumed by the subsequent training and evaluation steps.

In [ ]:
# ==========================================================
# Processing Step
# ==========================================================

processing_step = ProcessingStep(

    name="CustomerChurnPreprocessing",

    processor=script_processor,

    code=PREPROCESSING_SCRIPT,

    inputs=[

        ProcessingInput(

            input_name="raw_data",

            source=RAW_DATA_URI,

            destination="/opt/ml/processing/input",

        )

    ],

    outputs=[

        ProcessingOutput(

            output_name="train",

            source="/opt/ml/processing/train",

        ),

        ProcessingOutput(

            output_name="test",

            source="/opt/ml/processing/test",

        ),

    ],

)

print("=" * 60)
print("Processing Step Created Successfully")
print("=" * 60)

print(f"Script          : {PREPROCESSING_SCRIPT}")
print(f"Input Dataset   : {RAW_DATA_URI}")
print(f"Train Output    : /opt/ml/processing/train")
print(f"Test Output     : /opt/ml/processing/test")

print("=" * 60)

# Section 6 — Model Training

The training step launches a SageMaker Training Job using the `SKLearn` estimator.

During execution, SageMaker will:

1. Download the training dataset produced by the preprocessing step.
2. Install the packages listed in `requirements.txt`.
3. Execute `train.py`.
4. Train the Scikit-learn Pipeline with an XGBoost classifier.
5. Save the trained pipeline (`customer_churn_pipeline.pkl`) to `/opt/ml/model`.
6. Package the model as `model.tar.gz` and upload it to Amazon S3.

The resulting model artifact is used later by the evaluation step and, if approved, the model registration step.

In [ ]:
# ==========================================================
# Training Step
# ==========================================================

training_step = TrainingStep(

    name="CustomerChurnTraining",

    estimator=estimator,

    inputs={

        "train": TrainingInput(

            s3_data=processing_step.properties.ProcessingOutputConfig.Outputs[
                "train"
            ].S3Output.S3Uri,

            content_type="text/csv",

        )

    },

)

print("=" * 60)
print("Training Step Created Successfully")
print("=" * 60)

print(f"Training Script : {TRAIN_SCRIPT}")
print(f"Input Channel   : train")
print(f"Content Type    : text/csv")

print("=" * 60)

# Section 7 — Model Evaluation

After training completes, the model is evaluated using the held-out test dataset.

This stage performs the following:

- Downloads the trained model artifact.
- Loads the trained Scikit-learn pipeline.
- Runs predictions on the test dataset.
- Calculates evaluation metrics.
- Saves the metrics to `evaluation.json`.
- Exposes the evaluation report to the SageMaker Pipeline using a `PropertyFile`.

The metrics generated in this step are later used to decide whether the model should be registered.

# ============================
# Evaluation Property File
# ============================

evaluation_report = PropertyFile(

    name="EvaluationReport",

    output_name="evaluation",

    path=EVALUATION_FILE,

)

print("=" * 60)
print("Evaluation PropertyFile Created")
print("=" * 60)
print(f"Evaluation File : {EVALUATION_FILE}")
print("=" * 60)

In [ ]:
# ==========================================================
# Evaluation Step
# ==========================================================

evaluation_step = ProcessingStep(

    name="CustomerChurnEvaluation",

    processor=script_processor,

    code=EVALUATE_SCRIPT,

    property_files=[evaluation_report],

    inputs=[

        ProcessingInput(

            source=training_step.properties.ModelArtifacts.S3ModelArtifacts,

            destination="/opt/ml/model",

        ),

        ProcessingInput(

            source=processing_step.properties.ProcessingOutputConfig.Outputs[
                "test"
            ].S3Output.S3Uri,

            destination="/opt/ml/processing/test",

        ),

    ],

    outputs=[

        ProcessingOutput(

            output_name="evaluation",

            source="/opt/ml/processing/evaluation",

        )

    ],

)

print("=" * 60)
print("Evaluation Step Created Successfully")
print("=" * 60)

print(f"Evaluation Script : {EVALUATE_SCRIPT}")
print(f"Evaluation Output : {EVALUATION_FILE}")

print("=" * 60)

In [ ]:
# ==========================================================
# Model Metrics
# ==========================================================

model_metrics = ModelMetrics(

    model_statistics=MetricsSource(

        s3_uri=Join(

            on="/",

            values=[

                evaluation_step.properties.ProcessingOutputConfig.Outputs[
                    "evaluation"
                ].S3Output.S3Uri,

                EVALUATION_FILE,

            ],

        ),

        content_type="application/json",

    )

)

print("=" * 60)
print("ModelMetrics Object Created")
print("=" * 60)

# Section 8 — Model Registration & Pipeline Execution

The final stage assembles the SageMaker Pipeline.

Pipeline workflow:

1. Preprocessing
2. Training
3. Evaluation
4. Accuracy Check
5. Model Registration (if accuracy meets the threshold)
6. Pipeline Creation
7. Pipeline Execution

Only models whose evaluation accuracy is greater than or equal to the configured threshold are registered in the SageMaker Model Registry.

In [ ]:
# ==========================================================
# Accuracy Condition
# ==========================================================

accuracy_condition = ConditionGreaterThanOrEqualTo(

    left=JsonGet(

        step_name=evaluation_step.name,

        property_file=evaluation_report,

        json_path="classification_metrics.accuracy.value",

    ),

    right=accuracy_threshold,

)

print("=" * 60)
print("Accuracy Condition Created")
print("=" * 60)

In [ ]:
# ==========================================================
# Register Model Step
# ==========================================================

register_model_step = RegisterModel(

    name="RegisterCustomerChurnModel",

    estimator=estimator,

    model_data=training_step.properties.ModelArtifacts.S3ModelArtifacts,

    content_types=["text/csv"],

    response_types=["text/csv"],

    inference_instances=[INFERENCE_INSTANCE_TYPE],

    transform_instances=[INFERENCE_INSTANCE_TYPE],

    model_package_group_name=MODEL_PACKAGE_GROUP_NAME,

    approval_status=model_approval_status,

    model_metrics=model_metrics,

)

print("=" * 60)
print("RegisterModel Step Created")
print("=" * 60)

In [ ]:
# ==========================================================
# Condition Step
# ==========================================================

condition_step = ConditionStep(

    name="CheckModelAccuracy",

    conditions=[accuracy_condition],

    if_steps=[register_model_step],

    else_steps=[],

)

# ==========================================================
# Build Pipeline
# ==========================================================

pipeline = Pipeline(

    name=PIPELINE_NAME,

    parameters=[

        processing_instance_count,

        processing_instance_type,

        training_instance_type,

        accuracy_threshold,

        model_approval_status,

    ],

    steps=[

        processing_step,

        training_step,

        evaluation_step,

        condition_step,

    ],

    sagemaker_session=pipeline_session,

)

print("=" * 60)
print("Pipeline Object Created Successfully")
print("=" * 60)

In [ ]:
# ==========================================================
# Create / Update Pipeline
# ==========================================================

print("=" * 60)
print("Creating / Updating SageMaker Pipeline...")
print("=" * 60)

pipeline.upsert(
    role_arn=ROLE
)

print("Pipeline created successfully.")

# ==========================================================
# Execute Pipeline
# ==========================================================

execution = pipeline.start()

print("=" * 60)
print("Pipeline execution started.")
print("=" * 60)

print(f"Execution ARN: {execution.arn}")
